# Analiza dopasowania tekstu do języka na podstawie najczęstszych słów

Przeanalizuję w jakim stopniu język wiki pasuje do kilku szablonowych języków

**spis treści**
1. potrzebne importy i deklaracje stałych
2. wygenerowanie danych do word_count.json
3. przygotowanie danych dotyczących innych językóœ
4. funckja sprawdzająca poziom podobieństwa
5. przykładowe użycia



In [103]:
import pandas as pd
from wordfreq import zipf_frequency, top_n_list
import pdfplumber
import os
import json
import re
from collections import Counter
import pdfplumber 
import numpy as np
import matplotlib.pyplot as plt
BULAPEDIA_URL = "https://bulbapedia.bulbagarden.net/wiki/"
LANGUAGE_SOURCE_ENG = "data/eng.pdf" #articles from Wikipedia
LANGUAGE_SOURCE_DE = "data/de.pdf"
LANGUAGE_SOURCE_IT = "data/it.pdf"  


Zbieramy odpowiednie dane, scrapując artykuły

In [ ]:
!python3 wiki_scraper.py --url "https://bulbapedia.bulbagarden.net/wiki/" --clear-json --json-path "bad_score_word_counts.json" --count-words "Matrix_design_(TCG)"
!python3 wiki_scraper.py --url "https://bulbapedia.bulbagarden.net/wiki/" --clear-json --json-path "base_word_counts.json" --count-words "Pikachu"



Funkcja, która tworzy słownik postaci `"słowo" : jego oczekiwana liczba wystąpień na milion"` dla pewnego języka i `k` najczęściej potykanych słów. Funckja ta bazuje na wskaźniku zipf (czyli $\log_{10}(liczba wystąpień na miliard)$), który otrzymujemy z `wordfreq`, i za pomocą następującego wzoru: $freqinmilion = 10^{zipf} \cdot \frac{1}{1000} = 10^{zipf-3}$, funckja przeliczająca jest zaimplementowana w wikianalyzer 

In [105]:
def get_k_most_frequent_words(language_code, k=10):
    res = top_n_list(language_code, n=k)
    df = pd.DataFrame(res, columns=["word"])
    df["frequency_in_wiki_language"] = df["word"].apply(            
        lambda x: 10 ** (zipf_frequency(x, language_code) - 3) if (zipf_frequency(x, language_code) > 0) else 0
    )
    return dict(zip(df["word"], df["frequency_in_wiki_language"]))

Funkcja zliczająca słowa z PDF-a i zapisująca je do pliku JSON, analogiczna do count_words w wikicontroller.py


In [106]:
def count_words_from_pdf(pdf_file):
    if not os.path.exists(pdf_file):
        return f"Error: PDF file '{pdf_file}' not found."

    text = ""
    try:
        with pdfplumber.open(pdf_file) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += " " + page_text
    except Exception as e:
        return f"Error reading PDF file: {e}"

    if not text.strip():
        return "Error: No text found in the PDF."


    words = re.findall(r"\w+", text.lower())
    current_counts = Counter(words)
    wpm_dict = dict(current_counts)
    print(f"Total unique words counted: {len(wpm_dict)}")
    return wpm_dict

    

Funkcja, która tworzy słownik postaci `"słowo" : jego oczekiwana liczba wystąpień na milion"` dla danego pliku .json

In [107]:
def get_frequency_stats_from_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if data is {}:
        return f"Error: no data in {json_path}."

    total_words = sum(data.values())
    
    df = pd.DataFrame(data.items(), columns=["word", "count"])
    return dict(zip(df["word"], df["count"]))

Funkcja, która liczy "score" dopasowania języka - średnią ważoną dopasowania słowa, gdzie wagą jest jego częstotliwość w języku, daną wzorem $score = \frac{\sum_w{wpm_in_text[w]*wpm_in_lang[w]}}{1 000 000} = \frac{\sum{liczba wystąpień}*{częśtotliwość w języku}}{liczba słów w artykule}$. Dzięki temu długość artykułu nie wpływa na score.

In [108]:
def lang_confidence_normalized(wpm_text, wpm_lang):
    score = 0.0
    for w, count in wpm_text.items():
        score += count * wpm_lang.get(w, 0.0)
        
    total_count = sum(wpm_text.values())
    return score / total_count if total_count > 0 else 0.0

No i na koniec funckja zarządzająca wszystkim - dla konkretnego k 

In [ ]:
def manager(language_code, k):
    wpm_lang = get_k_most_frequent_words(language_code, k=k)
    score_text_eng = lang_confidence_normalized(wpm_text_eng, wpm_lang)
    score_text_de = lang_confidence_normalized(wpm_text_de, wpm_lang)
    score_text_it = lang_confidence_normalized(wpm_text_it, wpm_lang)
    score_bad_article = lang_confidence_normalized(wpm_article1, wpm_lang)
    score_good_article = lang_confidence_normalized(wpm_article2, wpm_lang)
    
    return {
        "score_text_eng": score_text_eng,
        "score_text_de": score_text_de,
        "score_text_it": score_text_it,
        "score_bad_article": score_bad_article,
        "score_good_article": score_good_article,
    }

Główna funkcja - sprawdza score dla różnych k i generuje wykres

In [ ]:
wpm_article1 = get_frequency_stats_from_json("base_word_counts.json")
wpm_article2 = get_frequency_stats_from_json("bad_score_word_counts.json")
wpm_text_eng = count_words_from_pdf(LANGUAGE_SOURCE_ENG)
wpm_text_de = count_words_from_pdf(LANGUAGE_SOURCE_DE)
wpm_text_it = count_words_from_pdf(LANGUAGE_SOURCE_IT)

In [ ]:
ks = [3, 10, 100, 10000]
languages = ["en", "it", "de"]

for language in languages:

    language_results = {}
    for k in ks:
        language_results[k] = manager(language, k)
        print(f"Completed analysis for language: {language}, k={k}")
        
    

    names = list(language_results[ks[0]].keys())     
    x = np.arange(len(ks))                 
    width = 0.1                            

    plt.figure(figsize=(10, 6))

    for i, name in enumerate(names):
        values = [language_results[k][name] for k in ks]
        
        # Rysujemy słupki dla tego pliku, przesunięte o (i * width)
        plt.bar(x + i * width, values, width, label=name)

    center_offset = width * (len(names) - 1) / 2
    plt.xticks(x + center_offset, ks)  

    plt.xlabel("k value")
    plt.ylabel("Score")
    plt.title(f"Results using {language} language model")
    plt.legend(title=language)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()



# odpowiedzi na pytania

1. Czy dobór języków miał duże znaczenie?
Pomijając przypadek języka używającego innego alfabetu, gdzie skuteczność byłaby bardzo dobra, odmiana ma duże znaczenie, a konkretnie liczba różnych form. Ten mechanizm zliczania każdą formę uznaje za unikalne słowo, a więc nawet jeśli pewne słowo, np "read" jest czesto używane w języku ang, to w polskim zastosowań "czytać" będzie niewiele, jednak gdyby zliczyć wszystkie odmiany i pochodne tego słowa razem, wynik byłby zupełnie inny. Może też okazać się że dany artykuł napisany jest w mniej standardowym czasie, co też wpłynie na skuteczność analizy. Poza tym języki z tej samej rodziny mają często wspólne sformułowania, tak więc np różnice między _score_ dla francuskiego i hiszpańskiego byłyby niewielkie, pewnie także dla skandynawskich języków porównywanych ze sobą.
2. Czy po wartościach language_words_with_frequency dla danych i najczęstszych słowach z języka danych widać, że w wybranym języku słowa często są odmieniane?
Zdecydowanie, w językach z wieloma formami najczęstszymi słowami są rodzajniki w różnych formach ale o tym samym znaczeniu, oraz zaimki, przyimki itp, natomiast w angielskim są to głównie spójniki, albo różne czasy czasownika - zdecydowanie różne od siebie znaczeniem.
3. Czy trudne było znalezienie takiego artykułu, dla którego wynik language_words_with_frequency jest jak najmniejszy w języku wiki? Czy to specyfika tego wiki?
Tutaj akurat specyfikacja wiki dużo ułatwiła, ponieważ jest dużo nazw własnych, a więc znalezienie krótkiego artykułu, w którym niewiele słów jest standardowych nie było trudne. Dla innych wiki takie działanie byłoby nieco trudniejsze, natomiast byłyby to pewnie niszowe, krótkie artykuły, być może zawierające dużo gwarowych sformułowań lub archaizmów.

# podsumowanie

Łatwo zauważyć, że im większe k tym lepsza dokładność. Widać też że między językami są różne zależności - niemieckie słowa prawie nie pojawiają sie w angielskich tekstach, w odróżnieniu od włoskich. Ciekawym było analizowanie wykresów dla różnych źródeł języków - nowoczesnych, kilkuwiecznych, naukowych i popularnych. Uważam że przy wystarczająco dużych próbkach danych dla języków takich jak angielski model ten jest całkiem skuteczny, jednakże dla języków z wieloma formami słów ma już mniejszą dokładność.